## Aufgabe: Vergleich der Schätzalgorithmen

In dieser Aufgabe werden drei Algorithmen zur Schätzung der Koeffizienten eines AR($N$)-Modells
(die **Autokorrelationsmethode**, **Autokovarianzmethode** und der **Burg-Algorithmus**) hinsichtlich ihrer **Verzerrung (Bias)** und **Varianz** miteinander verglichen.

Zusätzlich wird ein Verfahren betrachtet, das auf der Autokorrelationsmethode basiert, jedoch anstelle eines rechteckigen Fensters ein **Hamming-Fenster** zur Auswertung der Daten verwendet
(Python-Funktion: `numpy.hamming(·)`).

---

### Gegeben

Ein (All-Pole-)Übertragungssystem

$$
H(z) = \frac{1}{A(z)}
$$

mit

$$
\begin{aligned}
A(z) =;& 1 - 2.2346 z^{-1} + 2.3921 z^{-2} - 2.0324 z^{-3} \
&+ 1.2865 z^{-4} - 0.4954 z^{-5} + 0.2322 z^{-6} \
&- 0.2554 z^{-7} - 0.1848 z^{-8}
\end{aligned}
$$

der Ordnung $m = 8$, das durch einen stationären, mittelwertfreien, weißen Rauschprozess
$N(k)$ mit der Leistung

$$
\sigma_{\pi N}^2 = 1
$$

angeregt wird.

<div style="text-align:center">
  <img src="../CDR/l15sys.eps" width="70%">
</div>

---

### a)

Bestimmen Sie zunächst das **Pol-Nullstellen-Diagramm** des Systems.
In Python kann dies z. B. mit Funktionen aus `scipy.signal` erfolgen, indem Pole und Nullstellen explizit berechnet und grafisch dargestellt werden.

Für eine genauere Analyse bestimmen Sie die exakten **PARCOR-Koeffizienten**

$$
\gamma_i, \quad i = 1 \ldots 8 .
$$

Hierzu ist die **unverzerrte Autokorrelationsfolge**
$r_{\pi XX}(\kappa)$ mithilfe der Impulsantwort des Systems zu berechnen. Anschließend ist die exakte **Yule–Walker-Gleichung** (Gl. 9.3.17) unter Anwendung der **Levinson–Durbin-Rekursion** zu lösen, z. B. mit geeigneten Implementierungen aus `scipy` oder `statsmodels`.

---

### b)

Schätzen Sie das **Leistungsdichtespektrum (PDS)** des Prozesses $X(k)$ mithilfe eines AR($8$)-Modells und der oben beschriebenen Schätzverfahren.

Verwenden Sie **1000 Realisierungen** des stochastischen Prozesses $X(k)$, wobei jede Realisierung aus

$$
N = 64
$$

Abtastwerten besteht.

Um das Einschwingverhalten des Systems $H(z)$ zu berücksichtigen, erzeugen Sie mindestens

$$
2^{11} + 64
$$

Abtastwerte des Anregungsprozesses $N(k)$.
Das weiße Rauschen ist anschließend mit einer digitalen Filterimplementierung, z. B. `scipy.signal.lfilter`, zu filtern.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from helper_functions import lyw, lcov, lburg_algo
# BEGIN SOLUTION
# ##############################################################################
# ##  Solution: Spectral Estimation Methods                                   ##
# ##############################################################################

MA = np.array([1.0])
AR = np.array([1.0000, -2.2346, 2.3921, -2.0324, 1.2865, -0.4954, 0.2322, -0.2554, 0.1848])
NFFT = 2**6
n = len(AR) - 1

N = 64
N_vor = 2**11
mc_run = 2000  # Set to ~200 for faster testing

# --- Part A: Transient Response & Pole-Zero Plot ---

plt.figure(figsize=(6, 6))
z, p, k = signal.tf2zpk(MA, AR)
plt.scatter(np.real(p), np.imag(p), marker='x', color='red', label='Poles')

# Draw unit circle
uc = plt.Circle((0,0), 1, fill=False, color='black', linestyle='--')
plt.gca().add_patch(uc)

plt.grid(True)
plt.title(f'Poles of the AR({n}) Model')
plt.xlabel('Real Part')
plt.ylabel('Imaginary Part')
plt.axis('equal')
plt.show()

# Calculate exact PARCOR coefficients (via impulse response)
t, h_arma = signal.dimpulse((MA, AR, 1), n=512)
h_arma = h_arma[0].flatten()
rxx_all = np.correlate(h_arma, h_arma, mode='full')
mid = len(rxx_all) // 2
rxx = rxx_all[mid:mid+n+1]

# Use Levinson-Durbin for exact PARCOR
_, g_exact, _ = lyw(h_arma, n, lev=1)
print("Exact PARCOR Coefficients:")
for val in g_exact:
    print(f"{val:1.4f}")

# --- Part B: Monte Carlo Simulation ---

Omega_norm = np.linspace(0, 1, NFFT, endpoint=False)
_, h_ex = signal.freqz(MA, AR, worN=NFFT, whole=True)
Sxx_ex = np.abs(h_ex)**2

for run_idx in range(1, 5):
    Sxx_ar = np.zeros((mc_run, NFFT))
    Coeff = np.zeros((mc_run, len(AR)))
    
    print(f"Calculating Method {run_idx}...")
    
    for r in range(mc_run):
        noise = np.random.randn(20 * N + N_vor)
        x_all = signal.lfilter(MA, AR, noise)
        x = x_all[N_vor : N_vor + N + 1]
        
        if run_idx == 1:
            ARc, _, _ = lyw(x, n, 1)
            method_str = 'YW'
        elif run_idx == 2:
            x_ham = x * np.hamming(len(x))
            ARc, _, _ = lyw(x_ham, n, 1)
            method_str = 'YW-Ham.'
        elif run_idx == 3:
            ARc = lcov(x, n)
            method_str = 'Covar.'
        elif run_idx == 4:
            ARc, _ = lburg_algo(x, len(x), n)
            method_str = 'Burg'
            
        Coeff[r, :] = ARc
        _, h_est = signal.freqz([1], ARc, worN=NFFT, whole=True)
        Sxx_ar[r, :] = np.abs(h_est)**2

    Coeff_mean = np.mean(Coeff, axis=0)
    Coeff_std = np.std(Coeff, axis=0)
    C2sig_o = Coeff_mean + 2 * Coeff_std
    C2sig_u = Coeff_mean - 2 * Coeff_std
    Coeff_mse = np.mean(np.abs(AR - Coeff_mean)**2) # Mean Squared Error
    
    Sxx_mean = np.mean(Sxx_ar, axis=0)
    Sxx_2sigma = Sxx_mean + 2 * np.std(Sxx_ar, axis=0)

    # --- Plot 1: Coefficients ---
    plt.figure()
    indices = np.arange(n + 1)
    plt.stem(indices, AR, linefmt='r-', markerfmt='ro', basefmt=" ", label='Exact')
    plt.stem(indices, Coeff_mean, linefmt='b-', markerfmt='bx', basefmt=" ", 
             label=f'Mean / MSE={Coeff_mse:1.2e}')
    
    # 2-Sigma vertical lines
    for i in range(len(indices)):
        plt.plot([indices[i], indices[i]], [C2sig_u[i], C2sig_o[i]], 'g-')
    plt.plot(indices, C2sig_o, 'go', fillstyle='full', label=f'2σ-Bound / σ={np.mean(Coeff_std):1.2f}')
    plt.plot(indices, C2sig_u, 'go', fillstyle='full')
    
    plt.title(f'{method_str}-AR({n}) Coefficients; N={N}')
    plt.xlabel('i')
    plt.ylabel('Coefficients')
    plt.grid(True)
    plt.ylim([-4, 4])
    plt.legend()
    plt.show()

    # --- Plot 2: Power Spectral Density (PSD) ---
    plt.figure()
    plt.plot(Omega_norm, 10 * np.log10(Sxx_ex / np.max(Sxx_ex)), 'r--', label='Exact')
    plt.plot(Omega_norm, 10 * np.log10(Sxx_mean / np.max(Sxx_ex)), 'b-', label='Mean')
    plt.plot(Omega_norm, 10 * np.log10(Sxx_2sigma / np.max(Sxx_ex)), 'g-.', label='2σ-Bound')
    
    plt.ylabel(r'$S_{xx}(e^{j\Omega})$ in dB')
    plt.xlabel(r'$\Omega/2\pi$')
    plt.title(f'PSD of {method_str}-AR({n}); N={N}')
    plt.legend()
    plt.grid(True)
    plt.show()
# END SOLUTION

## Lösung: Vergleich der Schätzalgorithmen

### a)

Die Abbildung zeigt das Pol-Nullstellen-Diagramm des Systems $H(z)$. Es handelt sich um ein reellwertiges System, bei dem sich ein Polpaar bei

$$
\Omega / 2\pi \approx 0.05
$$

sehr nahe am Einheitskreis befindet.

Für den zugrunde liegenden AR(8)-Prozess ergeben sich bei Verwendung der exakten Yule–Walker-Gleichung die folgenden PARCOR-Koeffizienten:

$$
\gamma_1 = 0.9785,;
\gamma_2 = -0.9279,;
\gamma_3 = 0.1531,;
\gamma_4 = -0.6974,
$$

$$
\gamma_5 = 0.1815,;
\gamma_6 = -0.1563,;
\gamma_7 = -0.1631,;
\gamma_8 = -0.1848 .
$$

Es ist zu beachten, dass die ersten beiden PARCOR-Koeffizienten Werte nahe bei eins besitzen und damit das Verhalten des AR-Prozesses dominieren. Dieser AR-Prozess entspricht einem typischen Sprachsignal.

<div style="text-align:center">
  <img src="../pngs/ldsv9a04_a1.png">
</div>

---

### b)

Im Folgenden werden die wesentlichen Eigenschaften der bekannten spektralen Schätzverfahren zusammengefasst. Es wird ein Stichprobenintervall der Länge $N$ des Prozesses $X(k)$ angenommen. Für die Betrachtung wird die **Leistung des Prädiktionsfehlers** herangezogen.

Aufgrund der inversen Beziehung zwischen einem AR($n$)-Modell und einem Prädiktor ist die Minimierung der Prädiktionsfehlerleistung äquivalent zur Bestimmung der Parameter eines AR($n$)-Prozesses. Mit anderen Worten wird dem Prozess $X(k)$ ein Prädiktor zugeordnet, dessen Koeffizienten so gewählt werden, dass die Leistung des Prädiktionsfehlers minimiert wird und derjenigen des zugrunde liegenden weißen Rauschprozesses $N(k)$ entspricht.

---

#### Yule–Walker- / Autokorrelationsmethode

* Der Prozess $X(k)$ wird außerhalb des Stichprobenintervalls als Null angenommen (rechteckiges Fenster)
* Die Einschwingzeit des Prädiktors wird in die Minimierung einbezogen
* Die **verzerrte** Autokorrelationsfolge $\hat{r}_{\pi XX}(\kappa)$ wird verwendet
* Die Lösung basiert auf einer hermiteschen Toeplitz-Matrix
* Die Schätzung ist **verzerrt**, das AR-Modell ist jedoch stets **minimalphasig**

---

#### Autokovarianzmethode

* Der Prozess $X(k)$ wird außerhalb des Stichprobenintervalls als nicht definiert betrachtet
* Die Einschwingzeit des Prädiktors wird **nicht** in die Minimierung einbezogen
* Die **unverzerrte** Autokorrelationsfolge $r_{\pi XX}(\kappa)$ wird verwendet
* Es existiert keine Toeplitz-Matrix
* Die Schätzung ist **unverzerrt**, das AR-Modell ist jedoch nicht notwendigerweise **minimalphasig**

---

#### Burg-Algorithmus

* Gemeinsame Minimierung der Vorwärts- und Rückwärts-Prädiktionsfehlerleistung
* Der Prozess $X(k)$ wird außerhalb des Stichprobenintervalls als nicht definiert behandelt
* Die Einschwingzeit des Prädiktors wird in die Minimierung einbezogen
* Die Schätzung ist **unverzerrt** und das AR-Modell ist stets **minimalphasig**

---

#### Autokorrelationsmethode mit Hamming-Fenster

Anstelle eines rechteckigen Fensters wird auf den Prozess $X(k)$ ein Hamming-Fenster der gleichen Länge $N$ angewendet.

Anstatt die Korrelation zweier rechteckiger Fenster – die zu einem Bartlett-Fenster führt – zur Berechnung der verzerrten Autokorrelationsfolge zu verwenden, wird hier die Korrelation zweier Hamming-Fenster genutzt. Dies führt zu einer kleinen Verzerrung der Autokorrelationswerte $\hat{r}_{\pi XX}(\kappa)$ für kleine $\kappa$, während größere $\kappa$-Werte stärker gewichtet werden.

Zur Veranschaulichung dieses Verhaltens werden die unverzerrten Korrelationsresultate beider Fenster dargestellt. Dieses Verfahren wird häufig in der Sprachanalyse eingesetzt, da Sprachsignale typischerweise einen dominanten ersten PARCOR-Koeffizienten aufweisen, der im Wesentlichen durch $\hat{r}*{\pi XX}(0)$ und $\hat{r}*{\pi XX}(1)$ bestimmt wird.

<div style="text-align:center">
  <img src="../pngs/ldsv9a04fenster.png">
</div>

---

Die Abbildungen auf der linken Seite zeigen die geschätzten Koeffizienten, während die Abbildungen auf der rechten Seite die zugehörigen Leistungsdichtespektren darstellen.

<div style="text-align:center">
  <img src="../pngs/ldsv9a04_b11.png">
  <img src="../pngs/ldsv9a04_b12.png">
</div>

Die Ergebnisse der Autokorrelationsmethode („YW-AR(8)“) bestätigen das Vorhandensein einer Verzerrung dieser Schätzung. Beispielsweise liegen einige exakte Koeffizienten wie etwa $a_3$ außerhalb der $2\sigma$-Grenzen. Dieses Verhalten spiegelt sich auch in der relativ hohen Prädiktionsfehlerleistung

$$
E{e^2} = 1.18
$$

wider. Das Leistungsdichtespektrum wird nur schlecht approximiert, und der kritische Pol bei $\Omega / 2\pi \approx 0.05$ wird falsch modelliert.

---

<div style="text-align:center">
  <img src="../pngs/ldsv9a04_b21.png">
  <img src="../pngs/ldsv9a04_b22.png">
</div>

Bei Verwendung der mit einem Hamming-Fenster modifizierten Autokorrelationsmethode („YW-HAM-AR(8)“) wird für Prozesse mit großen PARCOR-Koeffizienten $\gamma_1$ und $\gamma_2$ eine deutliche Reduktion der Verzerrung beobachtet.

So liegt beispielsweise der Koeffizient $a_3$, der zuvor außerhalb der $2\sigma$-Grenzen lag, nun innerhalb dieser Grenzen. Die mittlere Prädiktionsfehlerleistung reduziert sich auf

$$
E{e^2} = 0.25 .
$$

Ein Nachteil der Hamming-Gewichtung ist eine erhöhte Standardabweichung $\sigma$. Das geschätzte Leistungsdichtespektrum zeigt jedoch eine deutliche Verbesserung durch den Einsatz des Hamming-Fensters.

---

<div style="text-align:center">
  <img src="../pngs/ldsv9a04_b31.png">
  <img src="../pngs/ldsv9a04_b32.png">
</div>

Eine unverzerrte Schätzung der Koeffizienten wird mit der Kovarianzmethode („cov-AR(8)“) erreicht. Die kleinen Abweichungen zwischen exakten und gemittelten Werten sind auf die endliche Anzahl von Realisierungen ($1000$) zurückzuführen. Die Standardabweichung verhält sich ähnlich wie bei der modifizierten Autokorrelationsmethode.

Damit stellt die Kovarianzmethode eine attraktive Alternative zur spektralen Schätzung dar, was auch durch die Approximation des Leistungsdichtespektrums bestätigt wird. Ihr Hauptnachteil liegt im fehlenden Zwang zur Minimalphasigkeit.

---

<div style="text-align:center">
  <img src="../pngs/ldsv9a04_b41.png">
  <img src="../pngs/ldsv9a04_b42.png">
</div>

Die obigen Abbildungen zeigen die Ergebnisse des Burg-Algorithmus („Burg-AR(8)“). Die gemeinsame Minimierung der Vorwärts- und Rückwärts-Prädiktionsfehler erweist sich als besonders geeignet. Die Verzerrungsfreiheit ist klar erkennbar, und das Leistungsdichtespektrum wird nahezu ebenso gut approximiert wie bei der Kovarianzmethode.

Hinsichtlich der Schätzvarianz verhält sich der Burg-Algorithmus für das hier betrachtete Beispiel ähnlich wie die Kovarianz- und die modifizierte Autokorrelationsmethode.
